# Imports

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import time
import pandas as pd
from pathlib import Path
import yaml
from sklearn.metrics import classification_report, mean_squared_error, r2_score

In [ ]:
# Load the shared configuration from config.yaml at the repository root
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config.yaml').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('config.yaml not found. Run this notebook from inside the repository.')
with open(REPO_ROOT / 'config.yaml') as f:
    CONFIG = yaml.safe_load(f)
PATHS = {name: REPO_ROOT / path for name, path in CONFIG['paths'].items()}

In [ ]:
if CONFIG['device'] != 'auto':
    device = torch.device(CONFIG['device'])
    print("Using device:", device)
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU")

else:
    device = torch.device("cpu")
    print("Using CPU")

# Configurations

In [ ]:
TASK = CONFIG['task']  # 'Suturing', 'Needle_Passing' or 'Knot_Tying'

# Configuration
NUM_SURGEONS = CONFIG['model']['num_surgeons']

#Gesture generation
GESTURE_VOCAB_SIZE = CONFIG['model']['gesture_vocab_size']  # 15 gestures + 1 MASK

# GRS generation
NUM_GRS_COMPONENTS = CONFIG['model']['num_grs_components']  # 6 components to predict -> 0-5

SEQ_LEN = CONFIG['data']['seq_len']
HIDDEN_DIM = CONFIG['model']['hidden_dim']
VISION_DIM = CONFIG['model']['vision_dim']
LANG_DIM = CONFIG['model']['lang_dim']
NUM_TIMESTEPS = CONFIG['model']['num_timesteps']
EPOCHS = CONFIG['training']['epochs']
BATCH_SIZE = CONFIG['training']['batch_size']
LR = CONFIG['training']['lr']
SEED = CONFIG['training']['seed']
LAMBDA = CONFIG['training']['grs_loss_weight'] # Weighting factor for GRS loss

In [ ]:
TRAIN_PATH = PATHS['features_dir'] / f"jigsaws_train_{TASK}.npz"
TEST_PATH = PATHS['features_dir'] / f"jigsaws_test_{TASK}.npz"
MODEL_SAVE_PATH = PATHS['models_dir'] / f"grs_model_{TASK}.pt"
CSV_SAVE_PATH = PATHS['training_records_dir'] / f"grs_training_record_{TASK}.csv"
RESULTS_SAVE_PATH = PATHS['results_dir'] / f"grs_results_{TASK}.csv"

os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
os.makedirs(os.path.dirname(CSV_SAVE_PATH), exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_SAVE_PATH), exist_ok=True)

In [ ]:
results = []
training_epochs_record = pd.DataFrame(columns=[
    'epoch', 
    'total_loss', 
    'gesture_loss', 
    'grs_loss', 
    'gesture_acc',
    'component_0_mae', 
    'component_1_mae', 
    'component_2_mae', 
    'component_3_mae', 
    'component_4_mae', 
    'component_5_mae', 
    'avg_grs_mae', 
    'train_time'
])

## Seed

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
random.seed(SEED)
np.random.seed(SEED)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

# Main

In [ ]:
# Forward corruption: transition matrices
q_matrix = np.zeros((NUM_TIMESTEPS, GESTURE_VOCAB_SIZE, GESTURE_VOCAB_SIZE))
for t in range(NUM_TIMESTEPS):
    alpha = 1 - (t + 1) / NUM_TIMESTEPS
    for i in range(GESTURE_VOCAB_SIZE):
        q_matrix[t, i] = (1 - alpha) / (GESTURE_VOCAB_SIZE - 1)
        q_matrix[t, i, i] = alpha
q_matrix = torch.tensor(q_matrix, dtype=torch.float32)

def corrupt(x0, t):
    B, T = x0.shape
    probs = q_matrix[t][:, x0.view(-1)]  # (V, B*T)
    probs = probs.permute(1, 0).view(B, T, GESTURE_VOCAB_SIZE)
    xt = torch.multinomial(probs.view(-1, GESTURE_VOCAB_SIZE), 1).view(B, T)
    return xt

class RealJIGSAWSDataset(Dataset):
    def __init__(self, npz_path):
        data = np.load(npz_path)
        self.vision_feat = torch.tensor(data["vision_feat"], dtype=torch.float32)
        self.lang_feat = torch.tensor(data["lang_feat"], dtype=torch.float32)
        self.gesture_seq = torch.tensor(data["gesture_seq"], dtype=torch.long)
        self.surgeon_id = torch.tensor(data["surgeon_id"], dtype=torch.long)
        self.timesteps = torch.randint(0, NUM_TIMESTEPS, (len(self.surgeon_id),))
        self.grs_components = torch.tensor(data["grs_components"], dtype=torch.long) # (1 to 5) -> (0 to 4) in preprocessing (6 components)

    def __len__(self):
        return len(self.surgeon_id)

    def __getitem__(self, idx):
        x0 = self.gesture_seq[idx]
        t = self.timesteps[idx]
        xt = corrupt(x0.unsqueeze(0), t).squeeze(0)
        return {
            "vision_feat": self.vision_feat[idx],
            "lang_feat": self.lang_feat[idx],
            "surgeon_id": self.surgeon_id[idx],
            "x0": x0,
            "xt": xt,
            "t": t,
            "grs_components": self.grs_components[idx]
        }

class TimeEmbedding(nn.Module):
    def __init__(self, num_timesteps, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(num_timesteps, hidden_dim)

    def forward(self, t):  # (B,)
        return self.embed(t)

class SurgeonEmbedding(nn.Module):
    def __init__(self, num_surgeons, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_surgeons, hidden_dim)

    def forward(self, surgeon_id):
        return self.embedding(surgeon_id)


class VLAEncoder(nn.Module):
    def __init__(self, vision_dim, lang_dim, hidden_dim):
        super().__init__()
        self.vision_proj = nn.Linear(vision_dim, hidden_dim)
        self.lang_proj = nn.Linear(lang_dim, hidden_dim)
        self.fuse = nn.Linear(2 * hidden_dim, hidden_dim)

    def forward(self, vision_feat, lang_feat):
        v = self.vision_proj(vision_feat)
        l = self.lang_proj(lang_feat)
        return self.fuse(torch.cat([v, l], dim=-1))

class DiffusionGRSModel(nn.Module):
    def __init__(self, gesture_vocab_size, num_grs_components, hidden_dim):
        super().__init__()
        self.token_embed = nn.Embedding(gesture_vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, 2*hidden_dim),
            nn.ReLU(),
            nn.Linear(2*hidden_dim, 1*hidden_dim),
            nn.ReLU(),
        )
        self.gesture_head = nn.Linear(hidden_dim, gesture_vocab_size)
        self.grs_heads = nn.ModuleList([
            nn.Linear(gesture_vocab_size * SEQ_LEN, 1) for _ in range(num_grs_components)
        ])

    def forward(self, xt, cond):
        x = self.token_embed(xt)
        cond = cond.unsqueeze(1).expand_as(x)
        x = self.net(x + cond)
        gesture_logits = self.gesture_head(x)
        B, T, V = gesture_logits.shape # batch, seqlen, gesturevocab
        gesture_logits_flattened = gesture_logits.view(B, -1) # B, T * V
        grs_logits = [grs_head(gesture_logits_flattened) for grs_head in self.grs_heads]
        return gesture_logits, grs_logits

class FullModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.vla = VLAEncoder(VISION_DIM, LANG_DIM, HIDDEN_DIM)
        self.surgeon_embed = SurgeonEmbedding(NUM_SURGEONS, HIDDEN_DIM)
        self.time_embed = TimeEmbedding(NUM_TIMESTEPS, HIDDEN_DIM)
        self.denoise = DiffusionGRSModel(GESTURE_VOCAB_SIZE, NUM_GRS_COMPONENTS, HIDDEN_DIM)

    def forward(self, xt, vision_feat, lang_feat, surgeon_id, t):
        cond = self.vla(vision_feat, lang_feat) + self.surgeon_embed(surgeon_id) + self.time_embed(t)
        return self.denoise(xt, cond)


In [ ]:
def train():
    
    train_set = RealJIGSAWSDataset(TRAIN_PATH)
    test_set = RealJIGSAWSDataset(TEST_PATH)

    print(f"Train set size: {len(train_set)}, Test set size: {len(test_set)}")

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

    model = FullModel()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    grs_criterion = nn.L1Loss()  # Mean Absolute Error for GRS components

    # send model to device
    model.to(device)

    # time calculation variable
    train_time = 0.0
    if device.type == 'cuda':
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        train_gesture_loss = 0.0
        train_grs_loss = 0.0
        train_gesture_acc = 0.0
        train_grs_maes = [0.0 for _ in range(NUM_GRS_COMPONENTS)]
        all_gesture_preds = []
        all_gesture_labels = []
        all_grs_preds = [[] for _ in range(NUM_GRS_COMPONENTS)]
        all_grs_labels = [[] for _ in range(NUM_GRS_COMPONENTS)]

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            
            # extract batch data
            vision_feat = batch["vision_feat"]
            lang_feat = batch["lang_feat"]
            surgeon_id = batch["surgeon_id"]
            x0 = batch["x0"]
            xt = batch["xt"]
            t = batch["t"]
            grs_components = batch["grs_components"]

            # send data to device
            vision_feat = vision_feat.to(device)
            lang_feat = lang_feat.to(device)
            surgeon_id = surgeon_id.to(device)
            x0 = x0.to(device)
            xt = xt.to(device)
            t = t.to(device)
            grs_components = grs_components.to(device)

            # measure time for training step
            if device.type == 'cuda':
                start.record()
            else:
                train_start_time = time.time()

            # actual training step
            gesture_logits, grs_logits = model(xt, vision_feat, lang_feat, surgeon_id, t)
            gesture_loss = criterion(gesture_logits.view(-1, GESTURE_VOCAB_SIZE), x0.view(-1))
            
            # GRS loss regression MAE
            grs_loss = 0.0
            for i in range(NUM_GRS_COMPONENTS):
                grs_pred = grs_logits[i].squeeze(-1)  # (B,)
                grs_loss += grs_criterion(grs_pred, grs_components[:, i].float())
            
            loss = gesture_loss + LAMBDA * grs_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # record end times
            if device.type == 'cuda':
                end.record()
                torch.cuda.synchronize()
                train_time += start.elapsed_time(end) / 1000.0  # convert ms to s
            else:
                train_end_time = time.time()
                train_time += (train_end_time - train_start_time)

            with torch.no_grad():
                # Gesture accuracy
                gesture_preds = torch.argmax(gesture_logits, dim=-1)
                correct = (gesture_preds == x0).float().mean().item()
                train_gesture_acc += correct

                all_gesture_preds.extend(gesture_preds.view(-1).cpu().numpy())
                all_gesture_labels.extend(x0.view(-1).cpu().numpy())

                # GRS component MAE
                for i in range(NUM_GRS_COMPONENTS):
                    grs_pred = grs_logits[i].squeeze(-1)  # (B,)
                    mae = torch.abs(grs_pred - grs_components[:, i].float()).mean().item()
                    train_grs_maes[i] += mae
                    
                    all_grs_preds[i].extend(grs_pred.cpu().numpy())
                    all_grs_labels[i].extend(grs_components[:, i].cpu().numpy())

            train_loss += loss.item()
            train_gesture_loss += gesture_loss.item()
            train_grs_loss += grs_loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_gesture_train_loss = train_gesture_loss / len(train_loader)
        avg_grs_train_loss = train_grs_loss / len(train_loader)
        avg_gesture_train_acc = train_gesture_acc / len(train_loader)
        avg_grs_train_maes = [mae / len(train_loader) for mae in train_grs_maes]
        avg_grs_train_mae = np.mean(avg_grs_train_maes)

        print(f"Epoch {epoch+1} | Total Loss: {avg_train_loss:.4f} | Gesture Loss: {avg_gesture_train_loss:.4f} | GRS Loss: {avg_grs_train_loss:.4f}")
        print(f"  Gesture Acc: {avg_gesture_train_acc:.4f}")
        print(f"  GRS Component MAEs: {[f'{mae:.4f}' for mae in avg_grs_train_maes]} | Avg GRS MAE: {avg_grs_train_mae:.4f}")
        print(f"  Train Time: {train_time:.2f}s")

        # Record training progress
        training_epochs_record.loc[len(training_epochs_record)] = [
            epoch+1, avg_train_loss, avg_gesture_train_loss, avg_grs_train_loss, avg_gesture_train_acc,
            avg_grs_train_maes[0], avg_grs_train_maes[1], avg_grs_train_maes[2], 
            avg_grs_train_maes[3], avg_grs_train_maes[4], avg_grs_train_maes[5],
            avg_grs_train_mae, train_time
        ]


    model.eval()
    with torch.no_grad():
        test_loss = 0.0
        test_gesture_loss = 0.0
        test_grs_loss = 0.0
        test_gesture_acc = 0.0
        test_grs_maes = [0.0 for _ in range(NUM_GRS_COMPONENTS)]
        all_gesture_preds = []
        all_gesture_labels = []
        all_grs_preds = [[] for _ in range(NUM_GRS_COMPONENTS)]
        all_grs_labels = [[] for _ in range(NUM_GRS_COMPONENTS)]

        for batch in tqdm(test_loader, desc="Testing", leave=False):
            # extract batch data
            vision_feat = batch["vision_feat"]
            lang_feat = batch["lang_feat"]
            surgeon_id = batch["surgeon_id"]
            x0 = batch["x0"]
            xt = batch["xt"]
            t = batch["t"]
            grs_components = batch["grs_components"]

            # send data to device
            vision_feat = vision_feat.to(device)
            lang_feat = lang_feat.to(device)
            surgeon_id = surgeon_id.to(device)
            x0 = x0.to(device)
            xt = xt.to(device)
            t = t.to(device)
            grs_components = grs_components.to(device)

            gesture_logits = None
            grs_logits = None

            B, L = xt.shape

            # iterative reverse diffusion 
            for step in range(NUM_TIMESTEPS):
                time_step = NUM_TIMESTEPS - 1 - step
                t_step_tensor = torch.full((x0.size(0),), time_step, dtype=torch.long, device=device)
                gesture_logits, grs_logits = model(xt, vision_feat, lang_feat, surgeon_id, t_step_tensor)
                gesture_probs = torch.softmax(gesture_logits, dim=-1)
                xt = torch.multinomial(gesture_probs.view(-1, GESTURE_VOCAB_SIZE), 1).view(B, L)
            
            gesture_loss = criterion(gesture_logits.view(-1, GESTURE_VOCAB_SIZE), x0.view(-1))
            
            # GRS loss: direct regression
            grs_loss = 0.0
            for i in range(NUM_GRS_COMPONENTS):
                grs_pred = grs_logits[i].squeeze(-1)  # (B,)
                grs_loss += grs_criterion(grs_pred, grs_components[:, i].float())
            
            loss = gesture_loss + LAMBDA * grs_loss

            # Gesture accuracy
            gesture_preds = torch.argmax(gesture_logits, dim=-1)
            correct = (gesture_preds == x0).float().mean().item()
            test_gesture_acc += correct

            all_gesture_preds.extend(gesture_preds.view(-1).cpu().numpy())
            all_gesture_labels.extend(x0.view(-1).cpu().numpy())

            # GRS component MAE
            for i in range(NUM_GRS_COMPONENTS):
                grs_pred = grs_logits[i].squeeze(-1)  # (B,)
                mae = torch.abs(grs_pred - grs_components[:, i].float()).mean().item()
                test_grs_maes[i] += mae
                
                all_grs_preds[i].extend(grs_pred.cpu().numpy())
                all_grs_labels[i].extend(grs_components[:, i].cpu().numpy())

            test_loss += loss.item()
            test_gesture_loss += gesture_loss.item()
            test_grs_loss += grs_loss.item()

        avg_test_loss = test_loss / len(test_loader)
        avg_gesture_test_loss = test_gesture_loss / len(test_loader)
        avg_grs_test_loss = test_grs_loss / len(test_loader)
        avg_gesture_test_acc = test_gesture_acc / len(test_loader)
        avg_grs_test_maes = [mae / len(test_loader) for mae in test_grs_maes]
        avg_grs_test_mae = np.mean(avg_grs_test_maes)
        # Calculate RMSE and R² for each GRS component
        grs_rmses = []
        grs_r2_scores = []
        for i in range(NUM_GRS_COMPONENTS):
            rmse = np.sqrt(mean_squared_error(all_grs_labels[i], all_grs_preds[i]))
            r2 = r2_score(all_grs_labels[i], all_grs_preds[i])
            grs_rmses.append(rmse)
            grs_r2_scores.append(r2)
        
        avg_grs_rmse = np.mean(grs_rmses)
        avg_grs_r2 = np.mean(grs_r2_scores)

        gesture_weighted_f1 = classification_report(all_gesture_labels, all_gesture_preds, labels=list(range(GESTURE_VOCAB_SIZE)), output_dict=True, zero_division=0)['weighted avg']['f1-score']

        print(f"\nTest Results:")
        print(f"  Total Loss: {avg_test_loss:.4f} | Gesture Loss: {avg_gesture_test_loss:.4f} | GRS Loss: {avg_grs_test_loss:.4f}")
        print(f"  Gesture Acc: {avg_gesture_test_acc:.4f} | Gesture F1: {gesture_weighted_f1:.4f}")
        print(f"  GRS Component MAEs: {[f'{mae:.4f}' for mae in avg_grs_test_maes]} | Avg GRS MAE: {avg_grs_test_mae:.4f}")
        print(f"  GRS Component RMSEs: {[f'{rmse:.4f}' for rmse in grs_rmses]} | Avg GRS RMSE: {avg_grs_rmse:.4f}")
        print(f"  GRS Component R²: {[f'{r2:.4f}' for r2 in grs_r2_scores]} | Avg GRS R²: {avg_grs_r2:.4f}")

        # Optional: print classification report for gestures
        print("\nGesture Classification Report:")
        print(classification_report(all_gesture_labels, all_gesture_preds, labels=list(range(GESTURE_VOCAB_SIZE)), zero_division=0))


        final_acc = avg_gesture_test_acc
        final_weighted_f1 = gesture_weighted_f1

    # Save results: gesture metrics (Table 15) and GRS metrics per component (Tables 12-14)
    result = {
        'kind': 'Gesture + GRS',
        'task': TASK,
        'type': 'from scratch',
        'gesture_accuracy': final_acc,
        'gesture_weighted_f1': final_weighted_f1,
        'avg_grs_mae': avg_grs_test_mae,
        'avg_grs_rmse': avg_grs_rmse,
        'avg_grs_r2': avg_grs_r2,
    }
    for i in range(NUM_GRS_COMPONENTS):
        result[f'grs_mae_{i}'] = avg_grs_test_maes[i]
        result[f'grs_rmse_{i}'] = grs_rmses[i]
        result[f'grs_r2_{i}'] = grs_r2_scores[i]
    results.append(result)

    return model, train_time

## Training

In [ ]:
model, train_time = train()
torch.save(model.state_dict(), MODEL_SAVE_PATH)

In [ ]:
print(f"train time: {train_time}")

# Save test results (read by notebooks/utils/pareto_front.ipynb and plots.ipynb)
pd.DataFrame(results).to_csv(RESULTS_SAVE_PATH, index=False)

training_epochs_record.to_csv(CSV_SAVE_PATH, index=False)